In [9]:
import cv2 as cv
import numpy as np

VIDEO_PATH = 'car_bike_track.mp4'

# poprawka (255 zamiast 256)
TOLERANCE_GREEN = [[0, 64], [180, 255], [180, 255]]
TOLERANCE_BLUE  = [[128, 255], [0, 64], [0, 64]]
TOLERANCE_RED   = [[0, 64], [0, 64], [212, 255]]

def detect_bounding_box(my_frame, my_tolerance):
    my_mask = np.zeros_like(my_frame).astype(np.uint8)
    map_blue = (my_tolerance[0][0] < my_frame[:, :, 0]) & (my_frame[:, :, 0] <= my_tolerance[0][1])
    map_green = (my_tolerance[1][0] < my_frame[:, :, 1]) & (my_frame[:, :, 1] <= my_tolerance[1][1])
    map_red = (my_tolerance[2][0] < my_frame[:, :, 2]) & (my_frame[:, :, 2] <= my_tolerance[2][1])
    my_mask[map_red & map_green & map_blue] = 255
    
    y, x = np.where(my_mask[:, :, 0].astype(bool))
    if len(x) == 0 or len(y) == 0:
        return my_mask, -1, -1, 0, 0
        
    xmin, xmax = int(x.min()), int(x.max())
    ymin, ymax = int(y.min()), int(y.max())
    return my_mask, xmin, ymin, max(1, xmax - xmin), max(1, ymax - ymin)

### Zadanie 1: Algorytm Mean Shift [2 punkty]
Klasa MeanShiftTracker inicjalizuje histogram HS (Hue + Saturation) dla wskazanego okna startowego i w kolejnych klatkach aktualizuje pozycję obiektu o stałym rozmiarze okna.

In [10]:
class MeanShiftTracker:
    def __init__(self, frame, track_window, color=(0, 255, 0)):
        self.track_window = track_window
        self.color = color
        x, y, w, h = track_window
        
        roi = frame[y:y + h, x:x + w]
        hsv_roi = cv.cvtColor(roi, cv.COLOR_BGR2HSV)
        mask = cv.inRange(hsv_roi, np.array((0., 60., 32.)), np.array((180., 255., 255.)))
        
        self.roi_hist = cv.calcHist([hsv_roi], [0, 1], mask, [180, 256], [0, 180, 0, 256])
        cv.normalize(self.roi_hist, self.roi_hist, 0, 255, cv.NORM_MINMAX)
        self.term_crit = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 100, 1)

    def update(self, frame):
        hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
        dst = cv.calcBackProject([hsv], [0, 1], self.roi_hist, [0, 180, 0, 256], 1)
        ret, self.track_window = cv.meanShift(dst, self.track_window, self.term_crit)
        return self.track_window

    def draw(self, frame):
        x, y, w, h = self.track_window
        cv.rectangle(frame, (x, y), (x + w, y + h), self.color, 2)


def run_meanshift():
    cap = cv.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret:
        print("Nie udało się otworzyć pliku wideo.")
        return

    _, x_g, y_g, w_g, h_g = detect_bounding_box(frame, TOLERANCE_GREEN)
    _, x_b, y_b, w_b, h_b = detect_bounding_box(frame, TOLERANCE_BLUE)
    _, x_r, y_r, w_r, h_r = detect_bounding_box(frame, TOLERANCE_RED)

    trackers = [
        MeanShiftTracker(frame, (x_g, y_g, w_g, h_g), (0, 255, 0)),
        MeanShiftTracker(frame, (x_b, y_b, w_b, h_b), (255, 0, 0)),
        MeanShiftTracker(frame, (x_r, y_r, w_r, h_r), (0, 0, 255))
    ]

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        for tracker in trackers:
            tracker.update(frame)
            tracker.draw(frame)

        cv.imshow('Zadanie 1: MeanShift', frame)
        if cv.waitKey(30) & 0xFF == 27:  # ESC kończy działanie
            break

    cap.release()
    cv.destroyAllWindows()

### Zadanie 2: Algorytm Cam Shift [2 punkty]
Funkcja cv.CamShift zwraca obrócony prostokąt (ret), który rysujemy przy użyciu cv.boxPoints oraz cv.polylines.

In [11]:
class CamShiftTracker:
    def __init__(self, frame, track_window, color=(0, 255, 0)):
        self.track_window = track_window
        self.color = color
        (x, y, w, h) = track_window
        
        roi = frame[y:y + h, x:x + w]
        hsv_roi = cv.cvtColor(roi, cv.COLOR_BGR2HSV)
        mask = cv.inRange(hsv_roi, np.array((0., 60., 32.)), np.array((180., 255., 255.)))
        
        self.roi_hist = cv.calcHist([hsv_roi], [0, 1], mask, [180, 256], [0, 180, 0, 256])
        cv.normalize(self.roi_hist, self.roi_hist, 0, 255, cv.NORM_MINMAX)
        self.term_crit = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 100, 1)

    def calculate(self, frame):
        hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
        dst = cv.calcBackProject([hsv], [0, 1], self.roi_hist, [0, 180, 0, 256], 1)
        ret, self.track_window = cv.CamShift(dst, self.track_window, self.term_crit)
        return ret

    def draw(self, frame, ret):
        pts = cv.boxPoints(ret)
        pts = pts.astype(np.int32) 
        cv.polylines(frame, [pts], True, self.color, 2)


def run_camshift():
    cap = cv.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret:
        return

    _, x_g, y_g, w_g, h_g = detect_bounding_box(frame, TOLERANCE_GREEN)
    _, x_b, y_b, w_b, h_b = detect_bounding_box(frame, TOLERANCE_BLUE)
    _, x_r, y_r, w_r, h_r = detect_bounding_box(frame, TOLERANCE_RED)

    trackers = [
        CamShiftTracker(frame, (x_g, y_g, w_g, h_g), (0, 255, 0)),
        CamShiftTracker(frame, (x_b, y_b, w_b, h_b), (255, 0, 0)),
        CamShiftTracker(frame, (x_r, y_r, w_r, h_r), (0, 0, 255))
    ]

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        for tracker in trackers:
            rot_rect = tracker.calculate(frame)
            tracker.draw(frame, rot_rect)

        cv.imshow('Zadanie 2: CamShift', frame)
        if cv.waitKey(30) & 0xFF == 27:
            break

    cap.release()
    cv.destroyAllWindows()

### Zadanie 3: Algorytm Lucas-Kanade Optical Flow [2 punkty]
Śledzenie rzadkiego przepływu optycznego (Sparse Optical Flow) dla punktów charakterystycznych wykrytych metodą Shi-Tomasi (cv.goodFeaturesToTrack).

In [12]:
def run_lucas_kanade():
    cap = cv.VideoCapture(VIDEO_PATH)
    
    feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)
    
    lk_params = dict(winSize=(15, 15), maxLevel=2,
                     criteria=(cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03))

    color = np.random.randint(0, 255, (100, 3))

    ret, old_frame = cap.read()
    if not ret:
        return
        
    old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)
    p0 = cv.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    mask = np.zeros_like(old_frame)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        if p0 is None or len(p0) < 5:
            p0 = cv.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
            mask = np.zeros_like(frame)

        p1, st, err = cv.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

            for i, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel()
                c, d = old.ravel()
                mask = cv.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % 100].tolist(), 2)
                frame = cv.circle(frame, (int(a), int(b)), 5, color[i % 100].tolist(), -1)

            img = cv.add(frame, mask)
            cv.imshow('Zadanie 3: Lucas-Kanade Optical Flow', img)

            old_gray = frame_gray.copy()
            p0 = good_new.reshape(-1, 1, 2)
        else:
            cv.imshow('Zadanie 3: Lucas-Kanade Optical Flow', frame)
            old_gray = frame_gray.copy()

        if cv.waitKey(30) & 0xFF == 27:
            break

    cap.release()
    cv.destroyAllWindows()

### Zadanie 4: Algorytm Farneback Optical Flow [2 punkty]
Program oblicza gęsty przepływ optyczny (Dense Optical Flow) i wizualizuje go na dwa sposoby naraz:Kodowanie kolorem (HSV) – kierunek ruchu to odcień (Hue), a prędkość to jasność (Value).Siatka wektorów ruchu – linie i punkty nałożone na klatkę wideo, pokazujące bezpośrednią estymację wektorów przesunięcia $(dx, dy)$.

In [13]:
def draw_flow_vectors(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2, -1).astype(int)
    fx, fy = flow[y, x].T
    
    lines = np.vstack([x, y, x + fx, y + fy]).T.reshape(-1, 2, 2)
    lines = np.int32(lines + 0.5)
    
    vis = img.copy()
    cv.polylines(vis, lines, 0, (0, 255, 0))
    for (x1, y1), (_x2, _y2) in lines:
        cv.circle(vis, (x1, y1), 1, (0, 255, 0), -1)
    return vis


def run_farneback():
    cap = cv.VideoCapture(VIDEO_PATH)
    ret, frame1 = cap.read()
    if not ret:
        return
        
    prvs = cv.cvtColor(frame1, cv.COLOR_BGR2GRAY)
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255

    while True:
        ret, frame2 = cap.read()
        if not ret:
            break
            
        next_frame = cv.cvtColor(frame2, cv.COLOR_BGR2GRAY)
        
        flow = cv.calcOpticalFlowFarneback(prvs, next_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        mag, ang = cv.cartToPolar(flow[..., 0], flow[..., 1])
        hsv[..., 0] = ang * 180 / np.pi / 2
        hsv[..., 2] = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX)
        bgr_hsv_flow = cv.cvtColor(hsv, cv.COLOR_HSV2BGR)

        vector_flow = draw_flow_vectors(frame2, flow, step=16)

        cv.imshow('Zadanie 4a: Farneback (Mapa HSV)', bgr_hsv_flow)
        cv.imshow('Zadanie 4b: Farneback (Wektory ruchu)', vector_flow)

        if cv.waitKey(30) & 0xFF == 27:
            break
        prvs = next_frame

    cap.release()
    cv.destroyAllWindows()

### Zadanie 5: Dwuwymiarowy Filtr Kalmana [2 punkty]
Klasa KalmanTracker2D realizuje dwuwymiarowy filtr Kalmana ze stanem $[x, y, v_x, v_y]^T$ (pozycja oraz prędkość w osiach X i Y) oraz pomiarem $[x, y]^T$ pochodzącym z funkcji detect_bounding_box. Śledzi wszystkie trzy obiekty, rysując zarówno rzeczywisty pomiar, jak i predykcję z filtru Kalmana.

In [ ]:
class KalmanTracker2D:
    def __init__(self, initial_x, initial_y, tolerance, color):
        self.tolerance = tolerance
        self.color = color
        self.trajectory = []

        self.kf = cv.KalmanFilter(4, 2)

        self.kf.measurementMatrix = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ], np.float32)

        self.kf.transitionMatrix = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], np.float32)

        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
        self.kf.errorCovPost = np.eye(4, dtype=np.float32)

        init_state = np.array([[np.float32(initial_x)],
                               [np.float32(initial_y)],
                               [0.],
                               [0.]], np.float32)
        self.kf.statePre = init_state.copy()
        self.kf.statePost = init_state.copy()

    def update(self, frame):
        prediction = self.kf.predict()
        pred_x, pred_y = int(prediction[0, 0]), int(prediction[1, 0])

        _, x, y, w, h = detect_bounding_box(frame, self.tolerance)

        if x != -1 and y != -1:
            meas_x = np.float32(x + w / 2.0)
            meas_y = np.float32(y + h / 2.0)
            measurement = np.array([[meas_x], [meas_y]], np.float32)

            estimated = self.kf.correct(measurement)
            est_x, est_y = int(estimated[0, 0]), int(estimated[1, 0])

            cv.rectangle(frame, (x, y), (x + w, y + h), self.color, 1)
            cv.circle(frame, (int(meas_x), int(meas_y)), 3, self.color, -1)
        else:
            est_x, est_y = pred_x, pred_y

        self.trajectory.append((est_x, est_y))
        if len(self.trajectory) > 1:
            pts = np.array(self.trajectory, np.int32).reshape((-1, 1, 2))
            cv.polylines(frame, [pts], False, self.color, 2)

        cv.circle(frame, (pred_x, pred_y), 10, self.color, 2)
        cv.drawMarker(frame, (pred_x, pred_y), self.color, cv.MARKER_CROSS, 12, 2)
        return pred_x, pred_y


def run_kalman_2d():
    cap = cv.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret:
        print("Nie można otworzyć wideo.")
        return

    _, xy_x, xy_y, wy, hy = detect_bounding_box(frame, TOLERANCE_GREEN)
    _, xb, yb, wb, hb = detect_bounding_box(frame, TOLERANCE_BLUE)
    _, xr, yr, wr, hr = detect_bounding_box(frame, TOLERANCE_RED)

    trackers = [
        KalmanTracker2D(xy_x + wy / 2, xy_y + hy / 2, TOLERANCE_GREEN, (0, 255, 255)),
        KalmanTracker2D(xb + wb / 2, yb + hb / 2, TOLERANCE_BLUE, (255, 0, 0)),
        KalmanTracker2D(xr + wr / 2, yr + hr / 2, TOLERANCE_RED, (0, 0, 255))
    ]

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        for tracker in trackers:
            tracker.update(frame)

        cv.imshow('Zadanie 5: Filtr Kalmana 2D', frame)
        if cv.waitKey(30) & 0xFF == 27:
            break

    cap.release()
    cv.destroyAllWindows()

In [ ]:
if __name__ == '__main__':
    print("Uruchamianie Zadania 1: MeanShift (esc)")
    run_meanshift()
    
    print("Uruchamianie Zadania 2: CamShift (esc)")
    run_camshift()
    
    print("Uruchamianie Zadania 3: Lucas-Kanade Optical Flow (esc)")
    run_lucas_kanade()
    
    print("Uruchamianie Zadania 4: Farneback Optical Flow (esc)")
    run_farneback()
    
    print("Uruchamianie Zadania 5: Filtr Kalmana 2D (esc)")
    run_kalman_2d()

Uruchamianie Zadania 1: MeanShift (naciśnij ESC, aby przejść dalej)...
Uruchamianie Zadania 2: CamShift (naciśnij ESC, aby przejść dalej)...
Uruchamianie Zadania 3: Lucas-Kanade Optical Flow (naciśnij ESC, aby przejść dalej)...
Uruchamianie Zadania 4: Farneback Optical Flow (naciśnij ESC, aby przejść dalej)...
Uruchamianie Zadania 5: Filtr Kalmana 2D (naciśnij ESC, aby zakończyć)...
